# Week 7 Report

**Date:** 2026-07-05 to 2026-07-10

## Goals for this week

- [ ] Complete analysis & compile results
- [ ] Begin Poster
- [ ] Begin Paper Draft

## What I did

After running my GSL model and DTR models both in full and just based on year and compiling the results, it became quickly apparent that I would need to run some smaller subset models to better identify what might be responsible for certain changes to mean of response and variance of response coefficients between models. In doing so, further results became apparent-- of which some made sense logically and some remain without a logical explanation. 

For example, running a DTR model based only on year found that variance in DTR has decreased 27.79% from 1960 to present, but running a DTR model with all 21 non-DTR coefficients included in the GSL model indicates that DTR variance has increased 77.6% in that same time period. Additionally, models with subsets of coefficients (grouped as climate indices, atmospheric/oceanic data, and temperature data) all find a decrease in variability, and models with specifically only year and one covariate from the list also all find a decrease in variability. Further testing may be necessary if we wish to explore this result further, though a brief foray into combining small sets of covariates from different subset groups (i.e. two covariates from the climate indices group with two covariates from the temperature data group) also failed to recreate the positive variability trend found in the full model.

However, on the whole, most of the results appear sound and have logical backing. Since compiling the results, I have successfully either shown that given results prove a hypothesis of ours, or been able to find published literature with a likely explanation for results unexpected by our hypotheses. A list of results can be found below in the "Results and figures" section. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import bambi as bmb
import arviz as az

def remove_nulls_simple(data_subset, variables):
        # Drop rows where any of the specified variable columns have nulls
        data_subset_cleaned = data_subset.dropna(subset=variables, ignore_index=True)
        # Subsets the cleaned dataframe back into only chosen variables
        vars_cleaned = data_subset_cleaned[variables]
        return vars_cleaned

def list_to_str(list_name):
    string = " + ".join(list_name)
    return string

pa_gs = pd.read_csv('/home/reu/project/data/pa_data.csv')
temps = pd.read_csv('/home/reu/project/data/pa_temps_supplement.csv')
pa_gs = pa_gs.merge(temps, on=["station_id", "year"], how="left")

pa_gs = pa_gs[pa_gs['year'] >= 1960]
pa_gs = pa_gs.drop(['station_id', 'last_spring_frost_date', 'first_fall_frost_date', 'station_name','state'], axis=1)

def bayesian_model_unstandardized(data, target_var_name, cov_list):

    cov_str = list_to_str(cov_list)
    cov_list_target_var = list(cov_list) + [target_var_name] + ['year']
    data = remove_nulls_simple(data, cov_list_target_var)
    
    formula = bmb.Formula(f"{target_var_name} ~ {cov_str} + year",
                      f"sigma ~ {cov_str} + year")
    model = bmb.Model(formula, data=data, dropna = True)
    output = model.fit(idata_kwargs={'log_likelihood':True})

    resid = data[target_var_name] - model.predict(output, inplace=False).posterior["mu"].mean(("chain", "draw")).values
    
    return model, output, resid

gsl_all_cov_list = ['dtr_annual','tmean_spring','tmean_fall','latitude','longitude','tmax_annual','oni_annual',
                'nao_annual','pna_annual','amo_annual','sst_north_atlantic','pwat_station','dewpoint_station',
                'soil_moisture_station','cloud_cover_station','evaporation_station', 'dtr_spring',
                'sst_gulf_mexico','pwat_southeast_us','dewpoint_2m_southeast_us','soil_moisture_southeast_us',
                'cloud_cover_southeast_us','evaporation_southeast_us']

dtr_all_cov_list = ['tmean_spring','tmean_fall','latitude','longitude','tmax_annual','oni_annual',
                'nao_annual','pna_annual','amo_annual','sst_north_atlantic','pwat_station','dewpoint_station',
                'soil_moisture_station','cloud_cover_station','evaporation_station',
                'sst_gulf_mexico','pwat_southeast_us','dewpoint_2m_southeast_us','soil_moisture_southeast_us',
                'cloud_cover_southeast_us','evaporation_southeast_us']

## Analysis

*(Add code and markdown cells as needed.)*

In [3]:
gsl_model, gsl_output, gsl_resid = bayesian_model_unstandardized(pa_gs, 'growing_season_length', gsl_all_cov_list)
az.summary(gsl_output).head(50)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.131       63          117.56 draws/s       0:00:17    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.167       31          111.19 draws/s       0:00:17    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.147       31          94.36 draws/s        0:00:21    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.159       31          107.76 draws/s       0:00:18    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 27 seconds.


,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
Intercept,127,68,18,240,2777,3143,1.00,1.3,0.94
dtr_annual,-9.52,0.44,-10,-8.8,2982,3124,1.00,0.0081,0.0057
tmean_spring,1.235,0.27,0.81,1.7,3684,3100,1.00,0.0044,0.0031
tmean_fall,5.753,0.278,5.3,6.2,3515,3323,1.00,0.0047,0.0033
latitude,-2.58,0.6,-3.5,-1.6,3691,3197,1.00,0.0098,0.0069
longitude,1.407,0.203,1.1,1.7,3255,3115,1.00,0.0036,0.0025
tmax_annual,3.63,0.46,2.9,4.4,3060,3130,1.00,0.0083,0.0059
oni_annual,0.11,0.51,-0.7,0.92,4756,3396,1.00,0.0074,0.0053
nao_annual,0.32,0.82,-1,1.7,3605,3033,1.00,0.014,0.0094
pna_annual,2.22,0.88,0.8,3.6,4723,3194,1.00,0.013,0.0091


In [4]:
dtr_model, dtr_output, dtr_resid = bayesian_model_unstandardized(pa_gs, 'dtr_annual', dtr_all_cov_list)
az.summary(dtr_output).head(50)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.208       63          151.22 draws/s       0:00:13    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.119       31          153.40 draws/s       0:00:13    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.164       31          95.90 draws/s        0:00:20    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.148       31          151.82 draws/s       0:00:13    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 26 seconds.


,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
Intercept,2.4,2.9,-2.2,7.1,3433,3066,1.00,0.05,0.035
tmean_spring,-0.5458,0.0106,-0.56,-0.53,4285,3364,1.00,0.00016,0.00012
tmean_fall,-0.5979,0.0103,-0.61,-0.58,4662,3482,1.00,0.00015,0.0001
latitude,0.0785,0.0276,0.035,0.12,5050,3220,1.00,0.00039,0.00027
longitude,-0.1301,0.0089,-0.14,-0.12,3498,3257,1.00,0.00015,0.0001
tmax_annual,1.3237,0.0139,1.3,1.3,4028,3403,1.00,0.00022,0.00015
oni_annual,0.067,0.0221,0.032,0.1,4206,3503,1.00,0.00034,0.00024
nao_annual,-0.215,0.0373,-0.27,-0.16,4003,3154,1.00,0.00059,0.00041
pna_annual,0.036,0.0422,-0.031,0.1,4831,2981,1.00,0.00061,0.00045
amo_annual,1.871,0.133,1.7,2.1,3940,3240,1.00,0.0021,0.0015


## Results and figures

*(Summarize key findings. Include any plots you made.)*

Models
* 1 - GSL modeled by DTR and Year Only
* 2 - GSL modeled by DTR, Year, and Mean Temperatures
* 3 - GSL modeled by DTR, Year, Mean Temps, and Max Temps
* 4 - GSL modeled by all 17 covariates
* 5 - Annual DTR modeled by Year only
* 6 - Annual DTR modeled by all 15 non-DTR covariates

Re-defining Trends in GSL and DTR
GSL
* Model 4 showed GSL as increasing at 0.21 days/yr ( +/- 0.038 days ? standard deviation)

DTR
* Model 5 (yr) showed DTR as decreasing at 0.012 deg C/yr
* Model 6 (yr w/cov) showed DTR as increasing at 0.009  deg C/yr
    * Not a single clue why this is the case
    * Modeled every single covariate individually and every single one showed a decrease in DTR

Defining direct impact of Annual DTR on GSL
* Model 4 associated a 1 degree increase in DTR with a 9.51 day reduction in growing season (+/- 0.46 days ? standard deviation)

GSL Variance
* Model 4 indicated a 26.7543% (+/- 8.24%? Standard Deviation) decrease in GSL Variability from 1960 to 2025 (Model 2 indicated 26.42% decrease, Model 1 indicated a 31.41% decrease)
* Model 4 indicated a 2.963% increase in GSL variability for 1 degree celsius increase in Mean Spring Temperature
    * Per standard deviation: 5.94% increase
    * Opposite of hypothesized expectation based on previous finding of less temperature variability when mean is higher
    * An increase in mean spring temperature in PA likely indicates an increase in mean spring temperature across much of the Northern Hemisphere/world → Arctic warming increases the frequency of both extreme cold and extreme snow events in the midlatitudes → anomalously warm arctic in the spring = more variable temperatures across PA in the spring → more variable GSL?
* Model 4 indicated a 1.39% decrease in GSL variability for each unit of longitude traveled West
    * Impact of ocean? If so, why would SST covariate not be significant at 95%?
* Model 4 indicated a 5.456% decrease in GSL variability for each 1 degree celsius increase in tmax_annual
    * Per standard deviation: 9.53% decrease
    * This does conform with expectation that higher temperatures mean less temperature variability
* Model 4 indicated a 13.066% increase in GSL variability for every 1 degree C increase in dtr_annual
* Model 4 indicated a 7.42% increase in GSL variability for every 2.74% (1 standard deviation) increase in Southeast US Cloud Cover
    * US Northeast anomalous spring cold events occur when the Northern Hemisphere Jet stream is further to the south, and therefore passing to the north of or directly over the US Southeast. As the anomalous cold occurs over the Northeast (causing a decrease in growing season overall and an increase in variability through the unpredictability of anomalous events), low-pressure systems are likely to pass over the US Southeast along the jet stream, increasing overall cloud cover in the region and thus causing this connection in the model.

DTR Variance
* Model 5 indicated a 27.79% decrease in DTR variability from 1960 to 2025
* Model 6 indicated a 77.6% increase in same time period
    * Not a single clue why this is the case
    * Modeled every single covariate individually and every single one saw decrease in DTR variability

Consideration of Mean Temperature on Impact of DTR to GSL variability
* Model 1 indicated that without considering temperature, for every degree increase in DTR, GSL Variance decreases 4.171% (becomes more predictable)
* Model 2 indicated that with mean temperatures are considered, for every degree increase in DTR, GSL Variance increases 3.169% (becomes less predictable)
* Model 4 indicated that with all covariables considered, for every degree increase in DTR, GSL variance increases 13.0658% (becomes much less predictable)
* Ran a new model (Model 3) after noticing an increase in sigma_dtr_annual (variance of responses for Annual DTR) between exploratory standardized model runs not including Maximum Annual Temperature and including Maximum Annual Temperature
* Model 3 indicated that, with mean and max temperatures considered, for every degree increase in DTR, GSL Variance increases 4.739%

Discussion
* Hypothesis was that given a warming mean temperature, DTR and GSL variability would generally decrease, since warmer temperatures are less variable
* GSL variability is decreasing → results agree with hypothesis → explain logic
* DTR variability is all kinds of weird → all models other than model 6 (all covariates together) show decrease in variability, model 6 shows huge increase in variability
* Hypothesis was that increasing DTR would lower GSL variability → results do not match hypothesis → what logical reasoning could this align with?
* Hypothesis was that DTR’s impact on GSL would be heavily tied to mean temperatures → results agree with hypothesis, but in the opposite manner that we expected… restate results → discuss potential logical explanations
* Hypothesis was that larger DTRs have shorter GSL → results match hypothesis (1 deg inc in DTR = 9.5 day dec in GSL) → restate logic

## Questions and blockers

* Struggling to visualize the connection between temperature and DTR's impact on GSL variance as a plot
* Looking for feedback on poster/paper draft

## Plan for next week

- [ ] Finish Poster
- [ ] Continue Drafting Paper